# Notebook 15 — Adaptive Gating Controller

This notebook extends the control-stack sequence by replacing a fixed robust outlier gate with an **adaptive innovation gate**.

The notebook exports:

- figures to `figures/adaptive_gating_controller/`
- summary data to `results/adaptive_gating_controller_summary.json`
- summary table to `results/adaptive_gating_controller_summary.csv`
- results markdown to `results/adaptive_gating_controller_summary.md`
- docs markdown to `docs/adaptive_gating_controller.md`

Core question:

> Can an adaptive gate learn disturbance conditions online and improve clean recovery after phase-lock failure?


In [ ]:
# Imports and output paths
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "15"
SLUG = "adaptive_gating_controller"
TITLE = "Adaptive Gating Controller"

FIG_DIR = Path("figures") / SLUG
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(9423)

THRESHOLD = 1 / np.sqrt(1**2 + 1**2)
N_BLOCKS = 80
T = np.arange(N_BLOCKS)
PULSE_T = np.linspace(0, 10, 400)

BURST_WINDOWS = [(28, 40), (52, 66)]
MODEL_MISMATCH_WINDOWS = [(28, 40), (52, 66)]

print("CGCS threshold:", THRESHOLD)
print("Output figure directory:", FIG_DIR)


In [ ]:
# Disturbance environment

def in_windows(i, windows):
    return any(a <= i <= b for a, b in windows)

# True drift signals: smooth baseline + disturbance windows
omega_true = (
    0.24*np.sin(0.30*T + 0.4)
    + 0.12*np.sin(0.89*T - 0.8)
    + 0.004*T
)
B_true = (
    0.002 + 0.0011*T
    + 0.012*np.sin(0.23*T)
    + 0.006*np.sin(0.61*T + 1.2)
)

# Model mismatch produces persistent command distortion.
for a, b in MODEL_MISMATCH_WINDOWS:
    omega_true[a:b+1] += np.linspace(0.08, -0.05, b-a+1)
    B_true[a:b+1] += np.linspace(0.02, 0.045, b-a+1)

# Measurements include noise and outlier bursts.
omega_noise = np.random.normal(0, 0.018, N_BLOCKS)
B_noise = np.random.normal(0, 0.006, N_BLOCKS)

omega_meas = omega_true + omega_noise
B_meas = B_true + B_noise

outlier_blocks = []
for a, b in BURST_WINDOWS:
    idx = np.arange(a, b+1)
    burst = np.random.choice(idx, size=max(3, len(idx)//4), replace=False)
    outlier_blocks.extend(burst.tolist())
    omega_meas[burst] += np.random.choice([-1, 1], len(burst)) * np.random.uniform(0.18, 0.45, len(burst))
    B_meas[burst] += np.random.choice([-1, 1], len(burst)) * np.random.uniform(0.035, 0.11, len(burst))

outlier_blocks = sorted(set(outlier_blocks))
print("Outlier blocks:", outlier_blocks)


In [ ]:
# Kalman / control helper functions

def scalar_kalman(z, q=7e-4, r=3e-4, x0=None):
    x = z[0] if x0 is None else x0
    P = 1.0
    xs = []
    for zi in z:
        P = P + q
        K = P / (P + r)
        x = x + K * (zi - x)
        P = (1 - K) * P
        xs.append(x)
    return np.array(xs)


def gated_scalar_kalman(z, q=7e-4, r=3e-4, gate=0.08, x0=None):
    x = z[0] if x0 is None else x0
    P = 1.0
    xs = []
    rejected = []
    innovations = []
    gates = []
    for i, zi in enumerate(z):
        P = P + q
        innovation = zi - x
        innovations.append(float(innovation))
        gates.append(float(gate))
        if abs(innovation) <= gate:
            K = P / (P + r)
            x = x + K * innovation
            P = (1 - K) * P
            rejected.append(False)
        else:
            rejected.append(True)
        xs.append(x)
    return np.array(xs), np.array(rejected), np.array(innovations), np.array(gates)


def adaptive_gate_scalar_kalman(z, q=7e-4, r=3e-4, base_gate=0.035, k=3.0, window=7, min_gate=0.025, max_gate=0.18, x0=None):
    x = z[0] if x0 is None else x0
    P = 1.0
    xs = []
    rejected = []
    innovations = []
    gates = []
    recent_abs = []
    for i, zi in enumerate(z):
        P = P + q
        innovation = float(zi - x)
        abs_innov = abs(innovation)
        if len(recent_abs) >= 3:
            arr = np.array(recent_abs[-window:])
            med = np.median(arr)
            mad = np.median(np.abs(arr - med)) + 1e-9
            gate = np.clip(med + k*1.4826*mad, min_gate, max_gate)
        else:
            gate = base_gate
        innovations.append(innovation)
        gates.append(float(gate))
        if abs_innov <= gate:
            K = P / (P + r)
            x = x + K * innovation
            P = (1 - K) * P
            rejected.append(False)
            recent_abs.append(abs_innov)
        else:
            rejected.append(True)
            # include clipped innovation magnitude so gate can adapt without fully trusting outlier
            recent_abs.append(min(abs_innov, max_gate))
        xs.append(x)
    return np.array(xs), np.array(rejected), np.array(innovations), np.array(gates)


def joint_kalman(z_omega, z_B, q_omega=8e-4, q_B=1e-5, q_cross=1.8e-5, r_omega=3e-4, r_B=4e-5):
    x = np.array([z_omega[0], z_B[0]], dtype=float)
    P = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_B]])
    R = np.diag([r_omega, r_B])
    H = np.eye(2)
    xs = []
    for zo, zb in zip(z_omega, z_B):
        P = P + Q
        z = np.array([zo, zb])
        y = z - H @ x
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x = x + K @ y
        P = (np.eye(2) - K @ H) @ P
        xs.append(x.copy())
    return np.array(xs)


def adaptive_gate_joint_kalman(z_omega, z_B, q_omega=8e-4, q_B=1e-5, q_cross=1.8e-5, r_omega=3e-4, r_B=4e-5,
                               base_gate_o=0.04, base_gate_b=0.018, k=3.0, window=7, min_gate_o=0.025, max_gate_o=0.20,
                               min_gate_b=0.008, max_gate_b=0.08):
    x = np.array([z_omega[0], z_B[0]], dtype=float)
    P = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_B]])
    R = np.diag([r_omega, r_B])
    H = np.eye(2)
    xs = []
    rejected = []
    gates_o, gates_b = [], []
    innov_o, innov_b = [], []
    recent_o, recent_b = [], []
    for zo, zb in zip(z_omega, z_B):
        P = P + Q
        z = np.array([zo, zb], dtype=float)
        y = z - H @ x
        io, ib = float(y[0]), float(y[1])
        if len(recent_o) >= 3:
            arr = np.array(recent_o[-window:])
            med = np.median(arr)
            mad = np.median(np.abs(arr - med)) + 1e-9
            gate_o = np.clip(med + k*1.4826*mad, min_gate_o, max_gate_o)
        else:
            gate_o = base_gate_o
        if len(recent_b) >= 3:
            arr = np.array(recent_b[-window:])
            med = np.median(arr)
            mad = np.median(np.abs(arr - med)) + 1e-9
            gate_b = np.clip(med + k*1.4826*mad, min_gate_b, max_gate_b)
        else:
            gate_b = base_gate_b
        accept_o = abs(io) <= gate_o
        accept_b = abs(ib) <= gate_b
        # partial update: if one channel rejected, update accepted channel only
        if accept_o and accept_b:
            H_use = np.eye(2); R_use = R; y_use = y
        elif accept_o and not accept_b:
            H_use = np.array([[1.0, 0.0]]); R_use = np.array([[r_omega]]); y_use = np.array([io])
        elif accept_b and not accept_o:
            H_use = np.array([[0.0, 1.0]]); R_use = np.array([[r_B]]); y_use = np.array([ib])
        else:
            H_use = None
        if H_use is not None:
            S = H_use @ P @ H_use.T + R_use
            Kmat = P @ H_use.T @ np.linalg.inv(S)
            x = x + Kmat @ y_use
            P = (np.eye(2) - Kmat @ H_use) @ P
        rejected.append(not (accept_o and accept_b))
        recent_o.append(min(abs(io), max_gate_o))
        recent_b.append(min(abs(ib), max_gate_b))
        gates_o.append(float(gate_o)); gates_b.append(float(gate_b))
        innov_o.append(io); innov_b.append(ib)
        xs.append(x.copy())
    return np.array(xs), np.array(rejected), np.array(innov_o), np.array(innov_b), np.array(gates_o), np.array(gates_b)


def moving_average(z, w=5):
    out = []
    for i in range(len(z)):
        out.append(np.mean(z[max(0, i-w+1):i+1]))
    return np.array(out)


In [ ]:
# Build policies
omega_ma = moving_average(omega_meas, 5)
B_ma = moving_average(B_meas, 5)

omega_scalar = scalar_kalman(omega_meas, q=8e-4, r=3e-4)
B_scalar = scalar_kalman(B_meas, q=1e-5, r=4e-5)

joint = joint_kalman(omega_meas, B_meas)
omega_joint, B_joint = joint[:,0], joint[:,1]

omega_robust, rej_robust_o, innov_robust_o, gates_robust_o = gated_scalar_kalman(omega_meas, q=8e-4, r=3e-4, gate=0.075)
B_robust, rej_robust_b, innov_robust_b, gates_robust_b = gated_scalar_kalman(B_meas, q=1e-5, r=4e-5, gate=0.035)
rej_robust = rej_robust_o | rej_robust_b

omega_adapt, rej_adapt_o, innov_adapt_o, gates_adapt_o = adaptive_gate_scalar_kalman(omega_meas, q=8e-4, r=3e-4)
B_adapt, rej_adapt_b, innov_adapt_b, gates_adapt_b = adaptive_gate_scalar_kalman(B_meas, q=1e-5, r=4e-5, base_gate=0.012, min_gate=0.008, max_gate=0.08)
rej_adapt = rej_adapt_o | rej_adapt_b

adapt_joint, rej_adapt_joint, innov_ajo, innov_ajb, gates_ajo, gates_ajb = adaptive_gate_joint_kalman(omega_meas, B_meas)
omega_adapt_joint, B_adapt_joint = adapt_joint[:,0], adapt_joint[:,1]

# constrained MPC-lite: smoothed command from adaptive joint, intentionally conservative
omega_mpc = 0.82*omega_adapt_joint + 0.18*moving_average(omega_adapt_joint, 6)
B_mpc = 0.82*B_adapt_joint + 0.18*moving_average(B_adapt_joint, 6)

policies = {
    "none": (omega_meas, B_meas),
    "moving_average": (omega_ma, B_ma),
    "fixed_joint_kalman": (omega_joint, B_joint),
    "robust_gated_kalman": (omega_robust, B_robust),
    "adaptive_gate_kalman": (omega_adapt, B_adapt),
    "adaptive_gate_joint_kalman": (omega_adapt_joint, B_adapt_joint),
    "constrained_mpc": (omega_mpc, B_mpc),
    "oracle": (omega_true, B_true),
}

rejection_series = {
    "robust_gated_kalman": rej_robust,
    "adaptive_gate_kalman": rej_adapt,
    "adaptive_gate_joint_kalman": rej_adapt_joint,
}

print("Policies:", list(policies.keys()))


In [ ]:
# Response model, CGCS metrics, and recovery metrics

def response_curve(omega, B):
    # bounded excited-state probability proxy
    return np.clip(0.5*(1 - np.cos((1.0 + omega) * PULSE_T)) + B, 0, 1)

target_curve = response_curve(0.0, 0.0)

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

def policy_curves(omega_series, B_series):
    return np.array([response_curve(o, b) for o, b in zip(omega_series, B_series)])

curves = {name: policy_curves(o, b) for name, (o, b) in policies.items()}
rmse = {name: np.sqrt(np.mean((c - target_curve)**2, axis=1)) for name, c in curves.items()}
cosines = {name: np.array([cosine_similarity(c, target_curve) for c in cs]) for name, cs in curves.items()}
margins = {name: cosines[name] - THRESHOLD for name in policies}


def failure_runs(cos_series, threshold=THRESHOLD):
    below = cos_series < threshold
    runs = []
    i = 0
    while i < len(below):
        if below[i]:
            start = i
            while i < len(below) and below[i]:
                i += 1
            end = i - 1
            recovery = i if i < len(below) else None
            runs.append((start, end, recovery))
        i += 1
    return runs


def recovery_metrics(name, post_window=6):
    cos = cosines[name]
    margin = margins[name]
    runs = failure_runs(cos)
    error_integral = float(np.sum(np.maximum(0, THRESHOLD - cos)))
    failure_count = len(runs)
    time_below = int(np.sum(cos < THRESHOLD))
    recoveries = [r for _, _, r in runs if r is not None]
    if recoveries:
        overs = []
        ring = []
        settling = []
        for r in recoveries:
            end = min(len(cos), r + post_window)
            seg = cos[r:end]
            marg = margin[r:end]
            overs.append(float(np.max(marg) - marg[0]) if len(marg) else 0.0)
            ring.append(float(np.mean(np.abs(np.diff(seg)))) if len(seg) > 1 else 0.0)
            # first position after recovery where following samples remain above 0.98 similarity
            settle = post_window
            for j in range(r, min(len(cos), r + post_window)):
                if np.all(cos[j:min(len(cos), j+3)] >= 0.98):
                    settle = j - r
                    break
            settling.append(settle)
        reentry_overshoot = float(np.mean(overs))
        ringing_score = float(np.mean(ring))
        settling_blocks = float(np.mean(settling))
    else:
        reentry_overshoot = 0.0
        ringing_score = 0.0
        settling_blocks = 0.0
    clean_recovery_score = (
        0.30*(1 - error_integral)
        - 0.15*failure_count
        - 0.05*time_below
        - 1.2*reentry_overshoot
        - 1.0*ringing_score
        - 0.03*settling_blocks
    )
    return {
        "policy": name,
        "failure_count": int(failure_count),
        "recovery_count": int(len(recoveries)),
        "time_below_threshold": int(time_below),
        "phase_alignment_error_integral": error_integral,
        "reentry_overshoot": reentry_overshoot,
        "ringing_score": ringing_score,
        "settling_blocks": settling_blocks,
        "mean_rmse": float(np.mean(rmse[name])),
        "max_rmse": float(np.max(rmse[name])),
        "min_cosine": float(np.min(cos)),
        "mean_cosine": float(np.mean(cos)),
        "clean_recovery_score": float(clean_recovery_score),
    }

summary_rows = [recovery_metrics(name) for name in policies]
summary_df = pd.DataFrame(summary_rows).sort_values("clean_recovery_score", ascending=False).reset_index(drop=True)
summary_df


In [ ]:
# Add adaptive-gate diagnostics
extra = {
    "robust_gated_kalman": {
        "gate_rejection_count": int(np.sum(rej_robust)),
        "adaptive_gate_mean": None,
        "adaptive_gate_std": None,
    },
    "adaptive_gate_kalman": {
        "gate_rejection_count": int(np.sum(rej_adapt)),
        "adaptive_gate_mean": float(np.mean(gates_adapt_o)),
        "adaptive_gate_std": float(np.std(gates_adapt_o)),
    },
    "adaptive_gate_joint_kalman": {
        "gate_rejection_count": int(np.sum(rej_adapt_joint)),
        "adaptive_gate_mean": float(np.mean(gates_ajo)),
        "adaptive_gate_std": float(np.std(gates_ajo)),
    },
}

summary_df["gate_rejection_count"] = summary_df["policy"].map(lambda p: extra.get(p, {}).get("gate_rejection_count", 0))
summary_df["adaptive_gate_mean"] = summary_df["policy"].map(lambda p: extra.get(p, {}).get("adaptive_gate_mean", np.nan))
summary_df["adaptive_gate_std"] = summary_df["policy"].map(lambda p: extra.get(p, {}).get("adaptive_gate_std", np.nan))
summary_df


In [ ]:
# Save figure helper

def save_fig(name):
    path_png = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    path_pdf = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.pdf"
    plt.savefig(path_png, dpi=220, bbox_inches="tight")
    plt.savefig(path_pdf, bbox_inches="tight")
    print("saved", path_png)


def shade_windows(ax):
    for a, b in BURST_WINDOWS:
        ax.axvspan(a, b, alpha=0.15, label="noise burst" if a == BURST_WINDOWS[0][0] else None)
    for a, b in MODEL_MISMATCH_WINDOWS:
        ax.axvspan(a, b, alpha=0.08, label="model mismatch" if a == MODEL_MISMATCH_WINDOWS[0][0] else None)


In [ ]:
# Figure 1: CGCS stability comparison
plt.figure(figsize=(14, 6))
ax = plt.gca()
ax.axhline(THRESHOLD, linestyle="--", label="45° threshold")
shade_windows(ax)
for name in policies:
    plt.plot(T, cosines[name], marker="o", linewidth=1.2, label=name)
plt.title("Adaptive gating controller: CGCS phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.02)
plt.legend(loc="lower left", ncol=2)
save_fig("cgcs_stability_comparison")
plt.show()


In [ ]:
# Figure 2: threshold margin
plt.figure(figsize=(14, 6))
ax = plt.gca()
ax.axhline(0, linestyle="--", label="threshold margin = 0")
shade_windows(ax)
for name in ["none", "moving_average", "fixed_joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "adaptive_gate_joint_kalman", "constrained_mpc"]:
    plt.plot(T, margins[name], marker="o", linewidth=1.2, label=name)
plt.title("Adaptive gating controller: threshold margin")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.legend(loc="lower left", ncol=2)
save_fig("threshold_margin")
plt.show()


In [ ]:
# Figure 3: adaptive gate trace
plt.figure(figsize=(14, 5))
ax = plt.gca()
shade_windows(ax)
plt.plot(T, gates_adapt_o, label="adaptive scalar Ω gate")
plt.plot(T, gates_ajo, label="adaptive joint Ω gate")
plt.plot(T, np.full_like(T, 0.075, dtype=float), linestyle="--", label="fixed robust Ω gate")
plt.title("Adaptive gating controller: gate trace")
plt.xlabel("calibration block")
plt.ylabel("Ω innovation gate")
plt.legend()
save_fig("gate_trace")
plt.show()


In [ ]:
# Figure 4: rejection events
plt.figure(figsize=(14, 3.8))
y_positions = {"robust_gated_kalman": 2, "adaptive_gate_kalman": 1, "adaptive_gate_joint_kalman": 0}
for name, y in y_positions.items():
    rej = rejection_series[name]
    xs = np.where(rej)[0]
    plt.scatter(xs, np.full_like(xs, y), marker="x", s=80, label=name)
for ob in outlier_blocks:
    plt.axvline(ob, alpha=0.08)
plt.yticks(list(y_positions.values()), list(y_positions.keys()))
plt.title("Adaptive gating controller: rejection events")
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.legend(loc="upper right")
save_fig("rejection_events")
plt.show()


In [ ]:
# Figure 5: phase error integral
plot_df = summary_df[summary_df.policy != "oracle"].sort_values("phase_alignment_error_integral")
plt.figure(figsize=(11, 6))
plt.bar(plot_df.policy, plot_df.phase_alignment_error_integral)
plt.xticks(rotation=25, ha="right")
plt.title("Adaptive gating controller: phase-error integral")
plt.ylabel("phase alignment error integral")
save_fig("phase_error_integral")
plt.show()


In [ ]:
# Figure 6: re-entry overshoot
plot_df = summary_df[summary_df.policy != "oracle"].sort_values("reentry_overshoot")
plt.figure(figsize=(11, 6))
plt.bar(plot_df.policy, plot_df.reentry_overshoot)
plt.xticks(rotation=25, ha="right")
plt.title("Adaptive gating controller: re-entry overshoot")
plt.ylabel("mean overshoot/drop after re-entry")
save_fig("reentry_overshoot")
plt.show()


In [ ]:
# Figure 7: ringing score
plot_df = summary_df[summary_df.policy != "oracle"].sort_values("ringing_score")
plt.figure(figsize=(11, 6))
plt.bar(plot_df.policy, plot_df.ringing_score)
plt.xticks(rotation=25, ha="right")
plt.title("Adaptive gating controller: ringing score")
plt.ylabel("mean |second difference| after re-entry")
save_fig("ringing_score")
plt.show()


In [ ]:
# Figure 8: clean recovery ranking
plt.figure(figsize=(12, 6))
plot_df = summary_df.sort_values("clean_recovery_score", ascending=False)
plt.bar(plot_df.policy, plot_df.clean_recovery_score)
plt.xticks(rotation=25, ha="right")
plt.title("Adaptive gating controller: clean recovery ranking")
plt.ylabel("clean recovery score")
save_fig("clean_recovery_ranking")
plt.show()


In [ ]:
# Figure 9: worst re-entry window
real_df = summary_df[summary_df.policy != "oracle"]
worst_policy = real_df.sort_values("clean_recovery_score").iloc[0].policy
worst_cos = cosines[worst_policy]
# choose window around worst threshold violation
worst_idx = int(np.argmin(worst_cos))
lo = max(0, worst_idx - 14)
hi = min(N_BLOCKS, worst_idx + 15)
plt.figure(figsize=(14, 6))
ax = plt.gca()
shade_windows(ax)
ax.axhline(THRESHOLD, linestyle="--", label="45° threshold")
for name in ["none", "moving_average", "fixed_joint_kalman", "robust_gated_kalman", "adaptive_gate_kalman", "adaptive_gate_joint_kalman", "constrained_mpc"]:
    plt.plot(T[lo:hi], cosines[name][lo:hi], marker="o", linewidth=1.2, label=name)
plt.title(f"Adaptive gating controller: worst re-entry window — worst policy: {worst_policy}")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.02)
plt.legend(loc="lower left", ncol=2)
save_fig("worst_reentry_window")
plt.show()


In [ ]:
# Export results JSON / CSV / markdown / docs
summary_path_json = RESULTS_DIR / f"{SLUG}_summary.json"
summary_path_csv = RESULTS_DIR / f"{SLUG}_summary.csv"
summary_path_md = RESULTS_DIR / f"{SLUG}_summary.md"
docs_path_md = DOCS_DIR / f"{SLUG}.md"

summary_records = summary_df.to_dict(orient="records")
with open(summary_path_json, "w") as f:
    json.dump({
        "notebook": f"{NOTEBOOK_ID}_{SLUG}",
        "threshold": THRESHOLD,
        "burst_windows": BURST_WINDOWS,
        "model_mismatch_windows": MODEL_MISMATCH_WINDOWS,
        "outlier_blocks": outlier_blocks,
        "summary": summary_records,
    }, f, indent=2)
summary_df.to_csv(summary_path_csv, index=False)

ranking_table = summary_df[[
    "policy", "clean_recovery_score", "failure_count", "time_below_threshold",
    "phase_alignment_error_integral", "reentry_overshoot", "ringing_score", "settling_blocks",
    "gate_rejection_count", "mean_rmse", "max_rmse", "min_cosine"
]].to_markdown(index=False)

figures = [
    ("CGCS phase-lock stability", "cgcs_stability_comparison", "Adaptive gates are compared against fixed filters, MPC, and oracle behavior under repeated disturbances."),
    ("Threshold margin", "threshold_margin", "Margin above the 45° CGCS gate shows when each policy loses and regains phase lock."),
    ("Gate trace", "gate_trace", "The adaptive innovation gate changes over calibration blocks rather than using one fixed rejection threshold."),
    ("Rejection events", "rejection_events", "Rejection events show when each gated policy ignores likely corrupted measurements."),
    ("Phase-error integral", "phase_error_integral", "Lower integral means less time and severity below phase-lock boundary."),
    ("Re-entry overshoot", "reentry_overshoot", "Lower overshoot means cleaner return after crossing back above threshold."),
    ("Ringing score", "ringing_score", "Lower ringing means fewer oscillations after re-entry."),
    ("Clean recovery ranking", "clean_recovery_ranking", "Composite score ranks policies by failure count, error integral, overshoot, ringing, and settling."),
    ("Worst re-entry window", "worst_reentry_window", "Zoomed view of the most difficult re-entry interval."),
]

figure_sections = []
for title, fname, desc in figures:
    figure_sections.append(f"""### {title}\n\n![{title}](../figures/{SLUG}/{NOTEBOOK_ID}_{SLUG}_{fname}.png)\n\n{desc}\n""")
figure_sections = "\n".join(figure_sections)

summary_md = f"""# Adaptive Gating Controller Results Summary

## Configuration

- Notebook: `{NOTEBOOK_ID}_{SLUG}`
- Blocks: {N_BLOCKS}
- Phase-lock threshold: {THRESHOLD:.4f}
- Burst windows: {BURST_WINDOWS}
- Model mismatch windows: {MODEL_MISMATCH_WINDOWS}
- Outlier blocks: {outlier_blocks}

## Policy Ranking

{ranking_table}

## Key Observations

- Adaptive gates use recent innovation residuals instead of one fixed outlier threshold.
- `adaptive_gate_joint_kalman` combines cross-channel state estimation with online measurement rejection.
- `robust_gated_kalman` remains a fixed-threshold baseline for disturbance rejection.
- `constrained_mpc` smooths command response but can sacrifice fast re-entry.

## Conclusion

Adaptive gating tests whether phase-lock recovery can be treated as an online disturbance-classification problem rather than a fixed-threshold filtering problem.
"""

with open(summary_path_md, "w") as f:
    f.write(summary_md)

docs_md = f"""# Adaptive Gating Controller (Notebook 15)

Notebook 15 compares fixed robust gating against adaptive innovation gating under disturbance windows and model-mismatch windows.

The adaptive gate is computed from recent innovation residuals:

```text
innovation = measured Ω - predicted Ω
adaptive_gate = median(|innovation|) + k · MAD(|innovation|)
```

## Results Summary

{ranking_table}

## Figures

{figure_sections}

## Key Takeaways

- Adaptive gating turns phase-lock recovery into an online rejection problem.
- The gate trace shows when a policy tightens or relaxes measurement acceptance.
- Clean recovery is scored by failure count, time below threshold, phase-error integral, re-entry overshoot, ringing, and settling time.

## Next Step

Notebook 16 can package the full control-stack comparison across Notebooks 05–15 into a paper-ready summary table and figure index.
"""

with open(docs_path_md, "w") as f:
    f.write(docs_md)

print("saved", summary_path_json)
print("saved", summary_path_csv)
print("saved", summary_path_md)
print("saved", docs_path_md)


In [ ]:
# Zip notebook outputs for local download from Colab
zip_path = f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in FIG_DIR.glob(f"{NOTEBOOK_ID}_{SLUG}_*"):
        zf.write(p)
    for p in [
        RESULTS_DIR / f"{SLUG}_summary.json",
        RESULTS_DIR / f"{SLUG}_summary.csv",
        RESULTS_DIR / f"{SLUG}_summary.md",
        DOCS_DIR / f"{SLUG}.md",
    ]:
        if p.exists():
            zf.write(p)
print("created", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Colab; zip is available locally:", zip_path)
